In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import os
import glob
import pandas as pd
import datetime
import numpy as np
import polars as pl
from pathlib import Path

# 주제 1. 국내 채용시장 및 채용 플랫폼 이용패턴 분석

# Company
- 기업 마스터 테이블
- 기업id, 기업 생성일, 마지막 기업 정보 수정일, 설립일, 구성원 수, 기업 팔로워 수, 기업 추천 수


| 컬럼명            | 데이터 타입 | 설명                              |
|-------------------|-------------|-----------------------------------|
| id                | INTEGER     | 기업 id (PK, company_id)         |
| cdate             | DATETIME    | 기업 생성일(UTC), 플랫폼에 기업 정보 등록한 시점 |
| mdate             | DATETIME    | 마지막 기업정보 수정일           |
| found_date        | DATE        | 설립일(KST)                       |
| employee_count    | STRING      | 구성원 수, 데이터 수집 시점에 집계된 숫자 |
| view_count        | INTEGER     | 기업 정보 조회수, 데이터 수집 시점에 집계된 숫자 |
| follow_count      | INTEGER     | 기업 팔로우 수, 데이터 수집 시점에 집계된 숫자 |
| reference_count   | INTEGER     | 기업 추천 수, 데이터 수집 시점에 집계된 숫자 |


In [2]:
DATA_DIR = Path(r"C:\Users\user\Desktop\playground\codeit_DA_13th\projects\intermediate1\midproject1\DBsources\topic1")
# 데이터 모아둔 폴더 경로

Company = pl.read_parquet(DATA_DIR / 'company.parquet')

Company = Company.with_columns(
    pl.col(pl.Utf8).replace(["", "nan", "NaN"], None)
)

In [3]:
Company.select(pl.all().null_count())

cdate,mdate,found_date,employee_count,view_count,follow_count,reference_count,company_uuid
u32,u32,u32,u32,u32,u32,u32,u32
0,0,24175,0,0,0,0,0


설립일 결측치가 많다

In [4]:
Company.select(pl.all().n_unique())

cdate,mdate,found_date,employee_count,view_count,follow_count,reference_count,company_uuid
u32,u32,u32,u32,u32,u32,u32,u32
41329,35488,5200,8,6830,458,63,41659


기업의 uuid는 모두 유니크하다

In [5]:
# 개수(count)가 많은 순서대로 정렬해서 보기
Company['employee_count'].value_counts(sort=True)

employee_count,count
str,u32
"""0명""",34978
"""1-10명""",3202
"""11-50명""",2589
"""51-200명""",673
"""201-500명""",125
"""501-1000명""",42
"""1001-5000명""",40
"""5000명 초과""",10


사원수는 0명- 아마 미수집 데이터 일듯한데 가장 많다

1. 전체 데이터
2. 0배제
3. 0만

순서로 분석 진행

In [6]:
# 사원수 0명 데이터 배제
filtered_company = Company.filter(pl.col("employee_count") != "0명")
# 사원수 0명 데이터만
employee_zero_company = Company.filter(pl.col("employee_count") == "0명")

print(employee_zero_company.head())

shape: (5, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ cdate      ┆ mdate      ┆ found_date ┆ employee_ ┆ view_coun ┆ follow_co ┆ reference ┆ company_u │
│ ---        ┆ ---        ┆ ---        ┆ count     ┆ t         ┆ unt       ┆ _count    ┆ uid       │
│ datetime[μ ┆ datetime[μ ┆ str        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ s]         ┆ s]         ┆            ┆ str       ┆ i32       ┆ i32       ┆ i32       ┆ str       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2022-06-09 ┆ 2022-06-09 ┆ null       ┆ 0명       ┆ 0         ┆ 0         ┆ 0         ┆ c46f2fa5- │
│ 04:08:40   ┆ 04:08:40   ┆            ┆           ┆           ┆           ┆           ┆ f940-40fb │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ -bd4a-aa0 │
│            ┆            ┆            ┆           ┆           ┆           ┆  

### 조회수, 팔로우 수, 추천 수

In [7]:
result = (
    Company.select(['view_count', 'follow_count', 'reference_count'])
    .describe()
    .filter(pl.col("statistic").is_in(['mean', 'max', 'min', '50%']))
    .with_columns(pl.all().exclude("statistic").cast(pl.Float64).round(1))
)
print(result)

shape: (4, 4)
┌───────────┬────────────┬──────────────┬─────────────────┐
│ statistic ┆ view_count ┆ follow_count ┆ reference_count │
│ ---       ┆ ---        ┆ ---          ┆ ---             │
│ str       ┆ f64        ┆ f64          ┆ f64             │
╞═══════════╪════════════╪══════════════╪═════════════════╡
│ mean      ┆ 1661.8     ┆ 10.2         ┆ 0.7             │
│ min       ┆ 0.0        ┆ 0.0          ┆ 0.0             │
│ 50%       ┆ 536.0      ┆ 0.0          ┆ 0.0             │
│ max       ┆ 575596.0   ┆ 2740.0       ┆ 249.0           │
└───────────┴────────────┴──────────────┴─────────────────┘


In [8]:
# 사원수 0명 배제시 
result = (
    filtered_company.select(['view_count', 'follow_count', 'reference_count'])
    .describe()
    .filter(pl.col("statistic").is_in(['mean', 'max', 'min', '50%']))
    .with_columns(pl.all().exclude("statistic").cast(pl.Float64).round(1))
)
print(result)

shape: (4, 4)
┌───────────┬────────────┬──────────────┬─────────────────┐
│ statistic ┆ view_count ┆ follow_count ┆ reference_count │
│ ---       ┆ ---        ┆ ---          ┆ ---             │
│ str       ┆ f64        ┆ f64          ┆ f64             │
╞═══════════╪════════════╪══════════════╪═════════════════╡
│ mean      ┆ 4865.0     ┆ 44.5         ┆ 3.1             │
│ min       ┆ 4.0        ┆ 0.0          ┆ 0.0             │
│ 50%       ┆ 2585.0     ┆ 9.0          ┆ 1.0             │
│ max       ┆ 575596.0   ┆ 2740.0       ┆ 249.0           │
└───────────┴────────────┴──────────────┴─────────────────┘


In [9]:
# 사원수 0명 only
result = (
    employee_zero_company.select(['view_count', 'follow_count', 'reference_count'])
    .describe()
    .filter(pl.col("statistic").is_in(['mean', 'max', 'min', '50%']))
    .with_columns(pl.all().exclude("statistic").cast(pl.Float64).round(1))
)
print(result)

shape: (4, 4)
┌───────────┬────────────┬──────────────┬─────────────────┐
│ statistic ┆ view_count ┆ follow_count ┆ reference_count │
│ ---       ┆ ---        ┆ ---          ┆ ---             │
│ str       ┆ f64        ┆ f64          ┆ f64             │
╞═══════════╪════════════╪══════════════╪═════════════════╡
│ mean      ┆ 1049.9     ┆ 3.6          ┆ 0.3             │
│ min       ┆ 0.0        ┆ 0.0          ┆ 0.0             │
│ 50%       ┆ 379.0      ┆ 0.0          ┆ 0.0             │
│ max       ┆ 180132.0   ┆ 2056.0       ┆ 90.0            │
└───────────┴────────────┴──────────────┴─────────────────┘


### 인기 그룹 확인

In [10]:
# 상위 10% (0.9)와 상위 5% (0.95) 지점 확인
thresholds = Company.select([
    pl.col("view_count").quantile(0.95).alias("view_95p"),
    pl.col("follow_count").quantile(0.95).alias("follow_95p"),
    pl.col("reference_count").quantile(0.95).alias("reference_95p")
])
print(thresholds)

shape: (1, 3)
┌──────────┬────────────┬───────────────┐
│ view_95p ┆ follow_95p ┆ reference_95p │
│ ---      ┆ ---        ┆ ---           │
│ f64      ┆ f64        ┆ f64           │
╞══════════╪════════════╪═══════════════╡
│ 6299.0   ┆ 42.0       ┆ 4.0           │
└──────────┴────────────┴───────────────┘


In [11]:
thresholds_f = filtered_company.select([
    pl.col("view_count").quantile(0.95).alias("view_95p"),
    pl.col("follow_count").quantile(0.95).alias("follow_95p"),
    pl.col("reference_count").quantile(0.95).alias("reference_95p")
])
print(thresholds)

shape: (1, 3)
┌──────────┬────────────┬───────────────┐
│ view_95p ┆ follow_95p ┆ reference_95p │
│ ---      ┆ ---        ┆ ---           │
│ f64      ┆ f64        ┆ f64           │
╞══════════╪════════════╪═══════════════╡
│ 6299.0   ┆ 42.0       ┆ 4.0           │
└──────────┴────────────┴───────────────┘


In [12]:
thresholds_z = employee_zero_company.select([
    pl.col("view_count").quantile(0.95).alias("view_95p"),
    pl.col("follow_count").quantile(0.95).alias("follow_95p"),
    pl.col("reference_count").quantile(0.95).alias("reference_95p")
])
print(thresholds)

shape: (1, 3)
┌──────────┬────────────┬───────────────┐
│ view_95p ┆ follow_95p ┆ reference_95p │
│ ---      ┆ ---        ┆ ---           │
│ f64      ┆ f64        ┆ f64           │
╞══════════╪════════════╪═══════════════╡
│ 6299.0   ┆ 42.0       ┆ 4.0           │
└──────────┴────────────┴───────────────┘


In [13]:
# 기준값 추출 (예: 위에서 나온 결과를 직접 변수에 할당)
view_limit = thresholds[0, "view_95p"]
follow_limit = thresholds[0, "follow_95p"]
reference_limit = thresholds[0, "reference_95p"]

Company = Company.with_columns(
    pl.when((pl.col("view_count") >= view_limit) | (pl.col("follow_count") >= follow_limit) | (pl.col("reference_count") >= reference_limit))
    .then(pl.lit("인기 기업"))
    .otherwise(pl.lit("일반 기업"))
    .alias("company_segment")
)

segment_analysis = (
    Company.group_by("company_segment")
    .agg([
        pl.len().alias("count"),
        pl.col("view_count").mean().alias("avg_view"),
        pl.col("follow_count").mean().alias("avg_follow"),
        pl.col("reference_count").mean().alias("avg_reference"),

        # 설립일(found_date)이 문자열이라면 숫자로 바꿔서 평균 설립연도 계산 가능
        # pl.col("found_date").cast(pl.Int64, strict=False).mean().alias("avg_found_year")
    ])
    .sort("avg_view", descending=True)
)

print(segment_analysis)

shape: (2, 5)
┌─────────────────┬───────┬──────────────┬────────────┬───────────────┐
│ company_segment ┆ count ┆ avg_view     ┆ avg_follow ┆ avg_reference │
│ ---             ┆ ---   ┆ ---          ┆ ---        ┆ ---           │
│ str             ┆ u32   ┆ f64          ┆ f64        ┆ f64           │
╞═════════════════╪═══════╪══════════════╪════════════╪═══════════════╡
│ 인기 기업       ┆ 3514  ┆ 10008.666477 ┆ 99.295959  ┆ 6.439385      │
│ 일반 기업       ┆ 38145 ┆ 892.840844   ┆ 1.95871    ┆ 0.216542      │
└─────────────────┴───────┴──────────────┴────────────┴───────────────┘


In [14]:
view_limit_f = thresholds_f[0, "view_95p"]
follow_limit_f = thresholds_f[0, "follow_95p"]
reference_limit_f = thresholds_f[0, "reference_95p"]

filtered_company = filtered_company.with_columns(
    pl.when((pl.col("view_count") >= view_limit_f) | (pl.col("follow_count") >= follow_limit_f) | (pl.col("reference_count") >= reference_limit_f))
    .then(pl.lit("인기 기업"))
    .otherwise(pl.lit("일반 기업"))
    .alias("company_segment")
)

segment_analysis = (
    filtered_company.group_by("company_segment")
    .agg([
        pl.len().alias("count"),
        pl.col("view_count").mean().alias("avg_view"),
        pl.col("follow_count").mean().alias("avg_follow"),
        pl.col("reference_count").mean().alias("avg_reference"),

        # 설립일(found_date)이 문자열이라면 숫자로 바꿔서 평균 설립연도 계산 가능
        # pl.col("found_date").cast(pl.Int64, strict=False).mean().alias("avg_found_year")
    ])
    .sort("avg_view", descending=True)
)

print(segment_analysis)

shape: (2, 5)
┌─────────────────┬───────┬──────────────┬────────────┬───────────────┐
│ company_segment ┆ count ┆ avg_view     ┆ avg_follow ┆ avg_reference │
│ ---             ┆ ---   ┆ ---          ┆ ---        ┆ ---           │
│ str             ┆ u32   ┆ f64          ┆ f64        ┆ f64           │
╞═════════════════╪═══════╪══════════════╪════════════╪═══════════════╡
│ 인기 기업       ┆ 610   ┆ 21937.837705 ┆ 282.834426 ┆ 16.578689     │
│ 일반 기업       ┆ 6071  ┆ 3149.506177  ┆ 20.582276  ┆ 1.795256      │
└─────────────────┴───────┴──────────────┴────────────┴───────────────┘


In [15]:
view_limit_z = thresholds_z[0, "view_95p"]
follow_limit_z = thresholds_z[0, "follow_95p"]
reference_limit_z = thresholds_z[0, "reference_95p"]

employee_zero_company = employee_zero_company.with_columns(
    pl.when((pl.col("view_count") >= view_limit_z) | (pl.col("follow_count") >= follow_limit_z) | (pl.col("reference_count") >= reference_limit_z))
    .then(pl.lit("인기 기업"))
    .otherwise(pl.lit("일반 기업"))
    .alias("company_segment")
)

segment_analysis = (
    employee_zero_company.group_by("company_segment")
    .agg([
        pl.len().alias("count"),
        pl.col("view_count").mean().alias("avg_view"),
        pl.col("follow_count").mean().alias("avg_follow"),
        pl.col("reference_count").mean().alias("avg_reference"),

        # 설립일(found_date)이 문자열이라면 숫자로 바꿔서 평균 설립연도 계산 가능
        # pl.col("found_date").cast(pl.Int64, strict=False).mean().alias("avg_found_year")
    ])
    .sort("avg_view", descending=True)
)

print(segment_analysis)

shape: (2, 5)
┌─────────────────┬───────┬─────────────┬────────────┬───────────────┐
│ company_segment ┆ count ┆ avg_view    ┆ avg_follow ┆ avg_reference │
│ ---             ┆ ---   ┆ ---         ┆ ---        ┆ ---           │
│ str             ┆ u32   ┆ f64         ┆ f64        ┆ f64           │
╞═════════════════╪═══════╪═════════════╪════════════╪═══════════════╡
│ 인기 기업       ┆ 5474  ┆ 3716.621666 ┆ 20.371027  ┆ 1.804165      │
│ 일반 기업       ┆ 29504 ┆ 555.190754  ┆ 0.496407   ┆ 0.0           │
└─────────────────┴───────┴─────────────┴────────────┴───────────────┘


1. 구성원 수와의 상관관계: 인기 기업들은 보통 employee_count가 큰 대기업인가요, 아니면 규모는 작아도 화제성이 높은 스타트업인가?

2. 추천수(reference_count)의 재발견: 평균이 0.7로 매우 낮지만, 인기 기업 그룹 내에서는 이 숫자가 유의미하게 올라가는지 확인해 보세요. 만약 인기 기업에서도 추천수가 낮다면 "유저들은 조용히 지켜만 볼 뿐(조회/팔로우), 직접적인 추천 액션에는 매우 인색하다"는 서비스 이용 패턴 결론을 내릴 수 있습니다.

변수간 선형성을 일부 보임, 낮은 숫자에 데이터가 몰려있어서 적당히 구분지어서 추가적으로 확인해볼 필요있음

In [16]:
Company['cdate'].min(),Company['cdate'].max()

(datetime.datetime(2013, 1, 10, 15, 47, 37),
 datetime.datetime(2023, 12, 31, 9, 59, 24))

기업등록 시점의 최소 최대값
- 등록시점에 따른 기업 평판 (조회,팔로우,추천) 숫자가 다른지 확인해볼 여지있음
- 설립일과 비교를 통해 설립시점으로 부터 해당 플랫폼에 등록한 시점의 차이의 분포를 통해 플랫폼의 평판을 확인해볼 수 있음

# CompanyAddress
- 기업 주소지 정보 테이블	기업id
- 기업 상위 주소지 정보, 주소지명(Ex. 본사)
1. 수도권 (서울 + 경기도): **구**까지 제공  
2. 광역시(수도권 제외): **시**까지 제공  
3. 그 외 지역: **도**까지 제공  
4. 해외 본사: **나라명**까지 제공  

| 컬럼명      | 데이터 타입 | 설명 |
|------------|------------|----------------------------------------------------------------|
| company_id | INTEGER    | 기업 id |
| address1   | STRING     | 주소(상위 정보) |
| name       | STRING     | 주소지명 (Ex: 본사, 연구소 등) |




In [17]:
DATA_DIR = Path(r"C:\Users\user\Desktop\playground\codeit_DA_13th\projects\intermediate1\midproject1\DBsources\topic1")
# 데이터 모아둔 폴더 경로

CompanyAddress = pl.read_parquet(DATA_DIR / 'companyaddress.parquet')

CompanyAddress = CompanyAddress.with_columns(
    pl.col(pl.Utf8).replace(["", "nan", "NaN"], None)
)

CompanyAddress.head()

name,company_uuid,address
str,str,str
null,"""0bf81092-9bb5-44c7-a8de-e1db10…","""서울 구로구"""
"""사무실""","""0bf81092-9bb5-44c7-a8de-e1db10…","""서울특별시 구로구"""
null,"""e8b24030-b0c5-4f07-9427-2d1e25…","""158 Hoàng Hoa Thám, Phường 12,…"
null,"""21743ac5-8957-4b2e-8f36-804d18…","""서울특별시 서초구"""
null,"""82be7598-af6f-4af9-86d5-f57ccb…","""서울특별시 종로구"""


In [18]:
CompanyAddress.select(pl.all().null_count())

name,company_uuid,address
u32,u32,u32
279,0,0


주소지명 결측치가 일부 존재

In [19]:
CompanyAddress['company_uuid'].value_counts(sort=True)

company_uuid,count
str,u32
"""39db3d5b-4e84-4c52-a68d-c097d8…",44
"""43408a24-5a2e-464a-b868-d50f39…",33
"""9de72f52-dc33-4400-a11e-1bd086…",30
"""e47b6773-6d74-4086-bd48-a95236…",21
"""47990a4b-3d21-431b-a465-cecbc4…",19
…,…
"""565e7499-1070-49c5-a3b9-a27e5c…",1
"""1b389776-4d77-47fd-83e2-d381ee…",1
"""ab5e6357-9176-4833-873b-214ce1…",1


1개 기업은 최대 44개의 주소를가짐

In [20]:
result = CompanyAddress.filter(pl.col("company_uuid") == "39db3d5b-4e84-4c52-a68d-c097d859542d")

print(result)

shape: (44, 3)
┌─────────────────────────────────┬─────────────────────────────────┬─────────────────────┐
│ name                            ┆ company_uuid                    ┆ address             │
│ ---                             ┆ ---                             ┆ ---                 │
│ str                             ┆ str                             ┆ str                 │
╞═════════════════════════════════╪═════════════════════════════════╪═════════════════════╡
│ 르호봇 강남 비즈니스 센터       ┆ 39db3d5b-4e84-4c52-a68d-c097d8… ┆ 서울특별시 강남구   │
│ 르호봇 광주 비즈니스 센터       ┆ 39db3d5b-4e84-4c52-a68d-c097d8… ┆ 광주광역시 서구     │
│ 르호봇 교대 비즈니스 센터       ┆ 39db3d5b-4e84-4c52-a68d-c097d8… ┆ 서울특별시 서초구   │
│ 르호봇 구의 비즈니스 센터       ┆ 39db3d5b-4e84-4c52-a68d-c097d8… ┆ 서울특별시 광진구   │
│ 르호봇 대구 비즈니스 센터       ┆ 39db3d5b-4e84-4c52-a68d-c097d8… ┆ 대구광역시 중구     │
│ …                               ┆ …                               ┆ …                   │
│ 르호봇 프라임 시청 비즈니스     ┆ 39db3d5b-4e84-4c52-a68d-c097d8… ┆ 서울특별시 중구

분점이 많은 경우 동일 uuid에 대해 여러 주소를 가짐

In [21]:
print(
    f"주소 존재하는 회사 id 수 : {CompanyAddress['company_uuid'].n_unique()} , "
    f"전체 회사 id 수 : {Company['company_uuid'].n_unique()}"
)

주소 존재하는 회사 id 수 : 14033 , 전체 회사 id 수 : 41659


모든 회사id에 대해 주소를 가지는 것은 아님

In [22]:
CompanyAddress['address'].value_counts(sort=True)

address,count
str,u32
"""서울특별시 강남구""",4093
"""서울특별시 서초구""",1369
"""서울특별시 마포구""",1191
"""성남시""",1050
"""서울특별시 성동구""",741
…,…
"""140 Paya Lebar Rd, AZ @ Paya L…",1
"""Via Laietana, 52, 08003 Barcel…",1
"""655 Knight Way, Stanford, CA 9…",1


주소가 다양하게 들어가 있음 정규표현식등을 활용해서 한글 / 그 외를 구분하고 살펴봐야 할 듯

Seoul 11건 No need 1건 검출됨

필터 씌우면 이렇게:
```
# 1. 한글 시작 여부 플래그 생성
CompanyAddress_start_with_kor = CompanyAddress.filter(
    (pl.col("address").str.contains(r"^[가-힣]")) | 
    (pl.col("address") == "Seoul") | 
    (pl.col("address") == "No need")
)
```

근데 서울은 구 단위까지 필터 씌워서 하는게 좋을 것 같고 no need는 사실상 결측치로 보는게 어떤지
해당 개체: Asiance Korea	4ae4a422-7628-4b44-b286-ae38069f099b	No need

In [23]:
import polars as pl

# 1. 한글 시작 여부 플래그 생성
CompanyAddress_start_with_kor = CompanyAddress.filter(
    (pl.col("address").str.contains(r"^[가-힣]"))
)

# 1-1. 통계 파악 (개수 및 비율)
total_count = CompanyAddress.height
filtered_count = CompanyAddress_start_with_kor.height
drop_count = total_count - filtered_count

print(f"전체 유니크 주소 수: {total_count}")
print(f"1차 필터링 후 주소 수: {filtered_count}")
print(f"제거된 (해외/기타) 주소 수: {drop_count}")
print(f"유지 비율: {(filtered_count / total_count * 100):.2f}%")

# 4. 필터링된 데이터 상위 확인
print("\n--- 1차 필터링 데이터 상위 10개 ---")
print(CompanyAddress_start_with_kor)

전체 유니크 주소 수: 17301
1차 필터링 후 주소 수: 17075
제거된 (해외/기타) 주소 수: 226
유지 비율: 98.69%

--- 1차 필터링 데이터 상위 10개 ---
shape: (17_075, 3)
┌─────────────────────────────────┬─────────────────────────────────┬───────────────────┐
│ name                            ┆ company_uuid                    ┆ address           │
│ ---                             ┆ ---                             ┆ ---               │
│ str                             ┆ str                             ┆ str               │
╞═════════════════════════════════╪═════════════════════════════════╪═══════════════════╡
│ null                            ┆ 0bf81092-9bb5-44c7-a8de-e1db10… ┆ 서울 구로구       │
│ 사무실                          ┆ 0bf81092-9bb5-44c7-a8de-e1db10… ┆ 서울특별시 구로구 │
│ null                            ┆ 21743ac5-8957-4b2e-8f36-804d18… ┆ 서울특별시 서초구 │
│ null                            ┆ 82be7598-af6f-4af9-86d5-f57ccb… ┆ 서울특별시 종로구 │
│ null                            ┆ 79e5b9d5-c92f-4cd4-954d-629d75… ┆ 서울특별시 종로구 │
│ …               

# CompanyFund
- 기업id, 투자유치일, 투자단계, 투자유치금, 통화(KRW/USD)

| 컬럼명       | 데이터 타입 | 설명       | Comment |
|-------------|------------|------------|----------------------------------------------------------------|
| company_id  | INTEGER    | 기업 id    |  |
| date        | DATE       | 투자일(KST) |  |
| round_type  | STRING     | 투자 단계  | 투자 단계 비공개, Seed, Angel, Series A, Series B, Series C, Series D, Pre-IPO, 해당없음 중 하나 |
| raised      | INTEGER    | 투자 금액   |  |
| currency    | STRING     | 원화/달러   | KRW, USD |


In [24]:
from pathlib import Path

CompanyFund = pl.read_parquet(DATA_DIR / 'companyfund.parquet')

# 널값 polars용 전처리
CompanyFund = CompanyFund.with_columns(
    pl.col(pl.Utf8).replace(["", "nan", "NaN"], None)
)

In [25]:
CompanyFund.select(pl.all().null_count())


fund_date,round_type,raised,currency,company_uuid
u32,u32,u32,u32,u32
0,0,0,103,0


In [26]:
CompanyFund.select(pl.all().n_unique())


fund_date,round_type,raised,currency,company_uuid
u32,u32,u32,u32,u32
2899,9,812,3,4785


전체회사 41659개 중 투자 관련 정보가 있는 회사는 4785개뿐

In [27]:
CompanyFund.select(pl.col("currency")).unique()

currency
str
null
"""USD"""
"""KRW"""


투자 금액은 KRW,USD, nan 타입

In [28]:
CompanyFund.select(pl.col("round_type")).unique()

round_type
str
"""Pre-IPO"""
"""Series B"""
"""Angel"""
"""Series A"""
"""해당없음"""
"""Seed"""
"""Series D"""
"""투자 단계 비공개"""
"""Series C"""


다양한 투자 단계에 대해 값들이 존재

In [29]:
CompanyFund = CompanyFund.with_columns(
    pl.col("raised").cast(pl.Int64)
)

result = (
    CompanyFund
    .filter(pl.col("raised") == 0)
    .group_by("round_type")
    .len()
    .sort("len", descending=True)
)

print(result)

shape: (9, 2)
┌──────────────────┬──────┐
│ round_type       ┆ len  │
│ ---              ┆ ---  │
│ str              ┆ u32  │
╞══════════════════╪══════╡
│ 투자 단계 비공개 ┆ 2090 │
│ Seed             ┆ 1104 │
│ Angel            ┆ 262  │
│ Series A         ┆ 227  │
│ 해당없음         ┆ 126  │
│ Series B         ┆ 33   │
│ Pre-IPO          ┆ 8    │
│ Series C         ┆ 6    │
│ Series D         ┆ 1    │
└──────────────────┴──────┘


다양한 round 단계에 대해 투자금액이 0원으로 (미공개?) 있는 경우가 있다. 투자 금액자체를 활용하기에는 누락 데이터가 많아 보임 / round 단계 정도로 파악해야함

# Job
- 채용공고 마스터 테이블
- 용공고id, 기업id, 생성일, 마지막 수정일, 채용분야, 채용시작일, 채용마감일, 경력, 연봉공개여부, 원격근무가능여부

| 컬럼명                | 데이터 타입 | 설명                 | Comment |
|----------------------|------------|----------------------|--------------------------------------------------------------|
| id                  | INTEGER    | 채용공고 id          | PK (job_id) |
| company_id          | INTEGER    | 기업 id              |  |
| cdate              | DATETIME   | 등록 날짜(KST)       |  |
| mdate              | DATETIME   | 마지막 수정일(KST)   | 수정하지 않았을 경우, cdate와 동일 |
| job_field          | STRING     | 업무분야             | SW 개발, 기획/PM, 마케팅, 디자인, 운영, 경영지원, 비즈니스, 투자, HW 개발, 게임 개발 중 하나 |
| career_type_string | STRING     | 경력 형태            | 인턴, 신입, 경력 중 1개 이상 선택 |
| start_date        | DATE       | 채용시작일(KST)      | 등록 날짜와 동일 |
| end_date          | DATE       | 채용마감일(KST)      |  |
| allow_remote      | INTEGER    | 원격근무 가능 여부   | 원격근무 가능 : 1, 원격근무 불가능 : 0 |
| can_show_salary   | INTEGER    | 연봉 공개 여부       | 연봉공개 : 1, 연봉비공개 : 0 |


In [30]:
import polars as pl

Job = pl.read_parquet(DATA_DIR / 'job.parquet')

Job = Job.with_columns(
    pl.col(pl.String).replace(["", "nan", "NaN"], None)
)

display(Job.head())

cdate,mdate,job_field,career_type_string,start_date,end_date,allow_remote,can_show_salary,job_uuid,company_uuid
datetime[μs],datetime[μs],str,str,str,str,i32,i32,str,str
2020-11-25 10:32:10,2020-11-25 10:32:10,"""SW 개발""","""신입,경력,인턴""",null,"""2020-11-25""",1,0,"""764292b0-53f8-4f14-ac83-6eafc7…","""daa34559-fc42-47f6-b5c6-cc6171…"
2020-12-03 16:24:16,2020-12-03 16:24:16,"""HW 개발""","""인턴""","""2020-12-03""","""2020-12-04""",1,0,"""b2b598d6-9ca3-4eb4-ac9c-bb57a6…","""daa34559-fc42-47f6-b5c6-cc6171…"
2019-06-13 08:47:25,2019-06-13 08:47:25,"""디자인""","""신입,경력,인턴""","""2019-06-13""","""2019-06-21""",0,0,"""017f4d7d-91a9-4ef7-9dd7-afabf7…","""d4e7e647-8f85-4e7f-8609-7173cc…"
2019-06-14 01:34:45,2019-07-02 00:28:35,"""마케팅""","""경력""","""2019-06-14""","""2019-07-15""",0,0,"""41707ef1-f733-4f58-bb29-0b2e61…","""d4e7e647-8f85-4e7f-8609-7173cc…"
2019-07-02 00:28:58,2019-07-16 13:03:08,"""운영""","""신입,경력""","""2019-07-02""","""2019-08-31""",0,0,"""d0aa0eef-65d2-4e00-8177-968a2d…","""d4e7e647-8f85-4e7f-8609-7173cc…"


In [31]:
Job.select(pl.all().null_count())


cdate,mdate,job_field,career_type_string,start_date,end_date,allow_remote,can_show_salary,job_uuid,company_uuid
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,43531,22718,0,0,0,0


cdate/mdate로 분류하는게 좋을듯

In [32]:
# 지원 공고 게시 일자 최소 최대
Job['cdate'].min(),Job['cdate'].max()

(datetime.datetime(2013, 1, 10, 15, 48, 38),
 datetime.datetime(2023, 12, 31, 4, 28, 30))

(선택) cdate기준으로 2022~2023년도의 데이터로 한정해서 분석 진행할 경우 아래 코드에서 Job_filtered를 사용해서 EDA 진행하면 됨. (로그 데이터와 맞추기 위함)

In [33]:
start_date = "2022-01-01"
end_date = "2023-12-31"

Job_filtered = Job.filter(
    pl.col("cdate").is_between(pl.date(2022, 1, 1), pl.date(2023, 12, 31))
)

# 통계 확인
total_count = Job.height
target_count = Job_filtered.height
ratio = (target_count / total_count) * 100

print(f"전체 데이터 개수: {total_count:,}건")
print(f"2022~2023년 데이터 개수: {target_count:,}건")
print(f"전체 대비 비중: {ratio:.2f}%")

# 최소/최대 일자 확인
print(f"수정된 데이터의 cdate 최소 일자: {Job_filtered['cdate'].min()}")
print(f"수정된 데이터의 cdate 최대 일자: {Job_filtered['cdate'].max()}")

전체 데이터 개수: 144,247건
2022~2023년 데이터 개수: 32,676건
전체 대비 비중: 22.65%
수정된 데이터의 cdate 최소 일자: 2022-01-01 04:01:38
수정된 데이터의 cdate 최대 일자: 2023-12-29 04:37:23


In [34]:
# 연도별 공고 갯수
year_counts = (
    Job
    .with_columns(pl.col("cdate").dt.year().alias("year"))
    .group_by("year")
    .len()
    .sort("len")
)

print(year_counts)

shape: (11, 2)
┌──────┬───────┐
│ year ┆ len   │
│ ---  ┆ ---   │
│ i32  ┆ u32   │
╞══════╪═══════╡
│ 2013 ┆ 652   │
│ 2014 ┆ 1339  │
│ 2015 ┆ 6142  │
│ 2023 ┆ 9133  │
│ 2016 ┆ 12034 │
│ …    ┆ …     │
│ 2018 ┆ 14493 │
│ 2019 ┆ 16551 │
│ 2020 ┆ 19057 │
│ 2022 ┆ 23544 │
│ 2021 ┆ 28082 │
└──────┴───────┘


In [35]:
# 공고별 직무 수
Job['job_field'].value_counts().sort('count', descending=True)

job_field,count
str,u32
"""SW 개발""",58923
"""마케팅""",19963
"""디자인""",19872
"""운영""",14611
"""기획/PM""",11719
"""비즈니스""",10459
"""경영지원""",6541
"""HW 개발""",1095
"""게임 개발""",644


In [36]:
Job.select(pl.all().null_count())

cdate,mdate,job_field,career_type_string,start_date,end_date,allow_remote,can_show_salary,job_uuid,company_uuid
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,43531,22718,0,0,0,0


시작,종료일이 없는 경우가 존재, 시작일보다 종료일이 더 없다 -> 시작일은 공고 올린날로 봐도 무방할듯

In [37]:
Job['company_uuid'].value_counts(sort=True)

company_uuid,count
str,u32
"""7e62c1ea-42ed-4654-b113-52157d…",3495
"""3d6730ee-b869-47b7-90c6-1e074e…",2266
"""5b00ba89-7ffc-403b-a35e-01371a…",1306
"""997dea23-cc15-4af7-8f9b-33dc2e…",1197
"""18046276-13b0-462b-bab5-7db92a…",883
…,…
"""41afc394-e2b6-4d7b-946b-d3c166…",1
"""0ce46d35-3a5b-4072-928b-f84985…",1
"""c430e9cc-ca56-4243-b1f7-5dbd7f…",1


한 회사에서 많은 공고를 올렸다. 추측해보면
- 특정 정기 채용때 다양한 포지션자리가 오픈
- 여러기간(년/월)에 걸쳐 채용 공고를 오픈

확인해볼만한 질문
- 기업들의 채용이 많은 시점은 언제인가?
    - job_filed,  career_type_string 구성에 따라
- 많은 공고가 동시에 열리는 기업의 특성은 무엇인가?
- 경력, 신입, 인턴에 따라 채용공고가 열리는 시점은 차이가 존재하는가?
- 원격 근무와 연봉 근무는 다른 컬럼들과 어떤 관계를 보이는가?
- 직무에 따른 공고의 특성차이가 존재하는지?

### 채용 시점

In [38]:
recruitment_trend = (
    Job.with_columns(pl.col("cdate").dt.truncate("1mo").alias("month"))
    .group_by(["month", "job_field", "career_type_string"])
    .agg(pl.count("job_uuid").alias("job_count"))
    .sort("month")
)

print(recruitment_trend)

shape: (5_313, 4)
┌─────────────────────┬───────────┬────────────────────┬───────────┐
│ month               ┆ job_field ┆ career_type_string ┆ job_count │
│ ---                 ┆ ---       ┆ ---                ┆ ---       │
│ datetime[μs]        ┆ str       ┆ str                ┆ u32       │
╞═════════════════════╪═══════════╪════════════════════╪═══════════╡
│ 2013-01-01 00:00:00 ┆ 마케팅    ┆ 신입,경력          ┆ 2         │
│ 2013-01-01 00:00:00 ┆ 디자인    ┆ 신입,경력          ┆ 4         │
│ 2013-01-01 00:00:00 ┆ 마케팅    ┆ 신입,경력,인턴     ┆ 1         │
│ 2013-01-01 00:00:00 ┆ SW 개발   ┆ 신입,인턴          ┆ 1         │
│ 2013-01-01 00:00:00 ┆ 운영      ┆ 신입,경력          ┆ 1         │
│ …                   ┆ …         ┆ …                  ┆ …         │
│ 2023-12-01 00:00:00 ┆ 기획/PM   ┆ 신입,경력          ┆ 4         │
│ 2023-12-01 00:00:00 ┆ SW 개발   ┆ 신입,인턴          ┆ 1         │
│ 2023-12-01 00:00:00 ┆ 비즈니스  ┆ 인턴               ┆ 3         │
│ 2023-12-01 00:00:00 ┆ 경영지원  ┆ 신입,경력          ┆ 4         │
│ 2023-1

### 동시에 많은 공고를 올린 기업들

In [39]:
# cdate를 '일(Day)' 단위로 절삭하여 그룹화
company_bursts = (
    Job.with_columns(
        pl.col("cdate").dt.date().alias("cdate_day")  # 시분초 제거 후 날짜만 추출
    )
    .group_by(["company_uuid", "cdate_day"])
    .agg([
        pl.count("job_uuid").alias("simultaneous_jobs"),
        pl.col("allow_remote").mean().alias("remote_ratio"),
        pl.col("can_show_salary").mean().alias("salary_visibility_ratio")
    ])
    .filter(pl.col("simultaneous_jobs") > 1)
    .sort("simultaneous_jobs", descending=True)
)

print(company_bursts.head(10))

shape: (10, 5)
┌─────────────────────────┬────────────┬───────────────────┬──────────────┬────────────────────────┐
│ company_uuid            ┆ cdate_day  ┆ simultaneous_jobs ┆ remote_ratio ┆ salary_visibility_rati │
│ ---                     ┆ ---        ┆ ---               ┆ ---          ┆ o                      │
│ str                     ┆ date       ┆ u32               ┆ f64          ┆ ---                    │
│                         ┆            ┆                   ┆              ┆ f64                    │
╞═════════════════════════╪════════════╪═══════════════════╪══════════════╪════════════════════════╡
│ 8850c5d0-c9f2-4346-be68 ┆ 2022-02-24 ┆ 64                ┆ 0.0          ┆ 0.0                    │
│ -2df598…                ┆            ┆                   ┆              ┆                        │
│ 8850c5d0-c9f2-4346-be68 ┆ 2021-12-21 ┆ 55                ┆ 0.0          ┆ 0.0                    │
│ -2df598…                ┆            ┆                   ┆              ┆ 

#### 통계적 분석 예시

In [40]:
# 전반적인 수치 요약
# 관심 있는 수치 컬럼만 선택해서 요약 데이터프레임 생성
summary = company_bursts.select([
    "simultaneous_jobs", 
    "remote_ratio", 
    "salary_visibility_ratio"
]).describe()

print(summary)

shape: (9, 4)
┌────────────┬───────────────────┬──────────────┬─────────────────────────┐
│ statistic  ┆ simultaneous_jobs ┆ remote_ratio ┆ salary_visibility_ratio │
│ ---        ┆ ---               ┆ ---          ┆ ---                     │
│ str        ┆ f64               ┆ f64          ┆ f64                     │
╞════════════╪═══════════════════╪══════════════╪═════════════════════════╡
│ count      ┆ 24637.0           ┆ 24637.0      ┆ 24637.0                 │
│ null_count ┆ 0.0               ┆ 0.0          ┆ 0.0                     │
│ mean       ┆ 3.325892          ┆ 0.117165     ┆ 0.594375                │
│ std        ┆ 2.802972          ┆ 0.30066      ┆ 0.470674                │
│ min        ┆ 2.0               ┆ 0.0          ┆ 0.0                     │
│ 25%        ┆ 2.0               ┆ 0.0          ┆ 0.0                     │
│ 50%        ┆ 2.0               ┆ 0.0          ┆ 1.0                     │
│ 75%        ┆ 4.0               ┆ 0.0          ┆ 1.0                     

In [41]:
# 컬럼 간 상관관계 계산
correlation = company_bursts.select([
    pl.corr("simultaneous_jobs", "remote_ratio").alias("jobs_vs_remote"),
    pl.corr("simultaneous_jobs", "salary_visibility_ratio").alias("jobs_vs_salary"),
    pl.corr("remote_ratio", "salary_visibility_ratio").alias("remote_vs_salary")
])

print(correlation)

shape: (1, 3)
┌────────────────┬────────────────┬──────────────────┐
│ jobs_vs_remote ┆ jobs_vs_salary ┆ remote_vs_salary │
│ ---            ┆ ---            ┆ ---              │
│ f64            ┆ f64            ┆ f64              │
╞════════════════╪════════════════╪══════════════════╡
│ -0.048814      ┆ -0.105628      ┆ -0.037693        │
└────────────────┴────────────────┴──────────────────┘


In [42]:
# 공고 규모에 따른 그룹핑 분석
stats_by_scale = (
    company_bursts.with_columns(
        pl.when(pl.col("simultaneous_jobs") <= 5).then(pl.lit("Small (2-5)"))
        .when(pl.col("simultaneous_jobs") <= 20).then(pl.lit("Medium (6-20)"))
        .otherwise(pl.lit("Large (21+)"))
        .alias("burst_scale")
    )
    .group_by("burst_scale")
    .agg([
        pl.count("company_uuid").alias("count_of_bursts"),
        pl.col("remote_ratio").mean().alias("avg_remote_ratio"),
        pl.col("salary_visibility_ratio").mean().alias("avg_salary_ratio")
    ])
    .sort("avg_remote_ratio")
)

print(stats_by_scale)

shape: (3, 4)
┌───────────────┬─────────────────┬──────────────────┬──────────────────┐
│ burst_scale   ┆ count_of_bursts ┆ avg_remote_ratio ┆ avg_salary_ratio │
│ ---           ┆ ---             ┆ ---              ┆ ---              │
│ str           ┆ u32             ┆ f64              ┆ f64              │
╞═══════════════╪═════════════════╪══════════════════╪══════════════════╡
│ Large (21+)   ┆ 123             ┆ 0.01723          ┆ 0.553621         │
│ Medium (6-20) ┆ 2446            ┆ 0.092803         ┆ 0.421073         │
│ Small (2-5)   ┆ 22068           ┆ 0.120422         ┆ 0.613811         │
└───────────────┴─────────────────┴──────────────────┴──────────────────┘


### 경력 형태(신입/경력/인턴)에 따른 채용 시점의 차이

In [43]:
#경력별 공고 개수 확인
Job['career_type_string'].value_counts(sort=True)

career_type_string,count
str,u32
"""경력""",74682
"""신입,경력""",36529
"""신입,경력,인턴""",20928
"""인턴""",4360
"""신입,인턴""",3373
"""신입""",3274
"""경력,인턴""",1101


In [44]:
# 경력 형태별 '월(Month)' 기준 공고 비중 분석
career_monthly_trend = (
    Job.with_columns(
        pl.col("cdate").dt.month().alias("month")
    )
    .group_by(["career_type_string", "month"])
    .agg(pl.count("job_uuid").alias("count"))
    .with_columns(
        # 각 경력 형태(신입/경력 등) 내에서 월별 비중 계산
        (pl.col("count") / pl.col("count").sum().over("career_type_string") * 100).alias("ratio_pct")
    )
    .sort(["career_type_string", "month"])
)

# 보기 편하게 출력 (Pivot 등을 활용하면 더 직관적입니다)
print(career_monthly_trend)

shape: (84, 4)
┌────────────────────┬───────┬───────┬───────────┐
│ career_type_string ┆ month ┆ count ┆ ratio_pct │
│ ---                ┆ ---   ┆ ---   ┆ ---       │
│ str                ┆ i8    ┆ u32   ┆ f64       │
╞════════════════════╪═══════╪═══════╪═══════════╡
│ 경력               ┆ 1     ┆ 6150  ┆ 8.234916  │
│ 경력               ┆ 2     ┆ 6514  ┆ 8.722316  │
│ 경력               ┆ 3     ┆ 6383  ┆ 8.546906  │
│ 경력               ┆ 4     ┆ 6326  ┆ 8.470582  │
│ 경력               ┆ 5     ┆ 6198  ┆ 8.299189  │
│ …                  ┆ …     ┆ …     ┆ …         │
│ 인턴               ┆ 8     ┆ 369   ┆ 8.463303  │
│ 인턴               ┆ 9     ┆ 470   ┆ 10.779817 │
│ 인턴               ┆ 10    ┆ 385   ┆ 8.830275  │
│ 인턴               ┆ 11    ┆ 348   ┆ 7.981651  │
│ 인턴               ┆ 12    ┆ 358   ┆ 8.211009  │
└────────────────────┴───────┴───────┴───────────┘


### 원격 근무 및 연봉 공개 여부와 다른 컬럼의 관계

In [45]:
# 직무별 원격 근무 및 연봉 공개 허용률
per_field_stats = (
    Job.group_by("job_field")
    .agg([
        pl.col("allow_remote").mean().alias("remote_allow_rate"),
        pl.col("can_show_salary").mean().alias("salary_open_rate"),
        pl.count("job_uuid").alias("total_jobs")
    ])
    .sort("total_jobs", descending=True)
)

print(per_field_stats)

shape: (10, 4)
┌───────────┬───────────────────┬──────────────────┬────────────┐
│ job_field ┆ remote_allow_rate ┆ salary_open_rate ┆ total_jobs │
│ ---       ┆ ---               ┆ ---              ┆ ---        │
│ str       ┆ f64               ┆ f64              ┆ u32        │
╞═══════════╪═══════════════════╪══════════════════╪════════════╡
│ SW 개발   ┆ 0.148397          ┆ 0.617942         ┆ 58923      │
│ 마케팅    ┆ 0.101037          ┆ 0.618644         ┆ 19963      │
│ 디자인    ┆ 0.138285          ┆ 0.661886         ┆ 19872      │
│ 운영      ┆ 0.071932          ┆ 0.668469         ┆ 14611      │
│ 기획/PM   ┆ 0.117672          ┆ 0.413858         ┆ 11719      │
│ 비즈니스  ┆ 0.092074          ┆ 0.658285         ┆ 10459      │
│ 경영지원  ┆ 0.08806           ┆ 0.373643         ┆ 6541       │
│ HW 개발   ┆ 0.106849          ┆ 0.497717         ┆ 1095       │
│ 게임 개발 ┆ 0.15528           ┆ 0.389752         ┆ 644        │
│ 투자      ┆ 0.145238          ┆ 0.435714         ┆ 420        │
└───────────┴──────────

# JobAddress
- 채용공고 주소지 정보 테이블
- 채용공고id, 근무지 상위 주소지 정보, 근무지명

In [46]:
JobAddress = pl.read_parquet(DATA_DIR / 'jobaddress.parquet')

JobAddress = JobAddress.with_columns(
    pl.col(pl.Utf8).replace(["", "nan", "NaN"], None)
)


In [47]:
jdu = JobAddress.select(pl.col('job_uuid').n_unique())
ju = Job.select(pl.col('job_uuid').n_unique())
print(f'채용공고 정보의 job uuid 수 : {ju} ,\n\n채용공고 주소지 정보의 job uuid 수  :  {jdu}')

채용공고 정보의 job uuid 수 : shape: (1, 1)
┌──────────┐
│ job_uuid │
│ ---      │
│ u32      │
╞══════════╡
│ 144247   │
└──────────┘ ,

채용공고 주소지 정보의 job uuid 수  :  shape: (1, 1)
┌──────────┐
│ job_uuid │
│ ---      │
│ u32      │
╞══════════╡
│ 4898     │
└──────────┘


모든 채용공고의 주소지 정보가 나와 있는것은 아니다. 대략 3%만 존재
하나의 job_uuid가 여러개의 jobaddress를 가지는 경우도 존재

In [48]:
target_uuid = 'bf207485-b1cd-4c69-bd91-6e4ac536b197'
temp = JobAddress.filter(pl.col("job_uuid") == target_uuid)

# 결과 확인
print(temp)

shape: (13, 3)
┌────────┬─────────────────────────────────┬───────────────────┐
│ name   ┆ job_uuid                        ┆ address           │
│ ---    ┆ ---                             ┆ ---               │
│ str    ┆ str                             ┆ str               │
╞════════╪═════════════════════════════════╪═══════════════════╡
│ 사무실 ┆ bf207485-b1cd-4c69-bd91-6e4ac5… ┆ 서울특별시 강남구 │
│ 사무실 ┆ bf207485-b1cd-4c69-bd91-6e4ac5… ┆ 서울특별시 강남구 │
│ 사무실 ┆ bf207485-b1cd-4c69-bd91-6e4ac5… ┆ 서울특별시 강남구 │
│ 사무실 ┆ bf207485-b1cd-4c69-bd91-6e4ac5… ┆ 서울특별시 강남구 │
│ 사무실 ┆ bf207485-b1cd-4c69-bd91-6e4ac5… ┆ 서울특별시 강남구 │
│ …      ┆ …                               ┆ …                 │
│ 사무실 ┆ bf207485-b1cd-4c69-bd91-6e4ac5… ┆ 서울특별시 강남구 │
│ 사무실 ┆ bf207485-b1cd-4c69-bd91-6e4ac5… ┆ 서울특별시 강남구 │
│ 사무실 ┆ bf207485-b1cd-4c69-bd91-6e4ac5… ┆ 서울특별시 강남구 │
│ 사무실 ┆ bf207485-b1cd-4c69-bd91-6e4ac5… ┆ 서울특별시 강남구 │
│ 사무실 ┆ bf207485-b1cd-4c69-bd91-6e4ac5… ┆ 서울특별시 강남구 │
└────────┴─────────────────────────────────┴───────────

- 모든행이 중복인 데이터가 존재, drop_duplicates()로 날리고 봐야함
- 지역,시기별 공고 차이 등에 대해서 분석해볼 수 있음

In [49]:
# 1. 모든 컬럼 기준 중복 제거
JobAddress_drop_duplicates = JobAddress.unique()

# 2. 만약 특정 uuid에 주소지가 여러 개일 수 있다면 유지, 
# 완전히 똑같은 행만 날리려면 위와 같이 unique()만 쓰면 됩니다.
print(f"원본 행 수: {len(JobAddress)}")
print(f"중복 제거 후 행 수: {len(JobAddress_drop_duplicates)}")

원본 행 수: 4997
중복 제거 후 행 수: 4898


In [50]:
target_uuid = 'bf207485-b1cd-4c69-bd91-6e4ac536b197'
temp = JobAddress_drop_duplicates.filter(pl.col("job_uuid") == target_uuid)

# 결과 확인
print(temp)

shape: (1, 3)
┌────────┬─────────────────────────────────┬───────────────────┐
│ name   ┆ job_uuid                        ┆ address           │
│ ---    ┆ ---                             ┆ ---               │
│ str    ┆ str                             ┆ str               │
╞════════╪═════════════════════════════════╪═══════════════════╡
│ 사무실 ┆ bf207485-b1cd-4c69-bd91-6e4ac5… ┆ 서울특별시 강남구 │
└────────┴─────────────────────────────────┴───────────────────┘


# JobBookmark
- 채용 북마크 트랜잭션 테이블
- 유저id, 채용공고id, 북마크일시


| 컬럼명      | 데이터 타입 | 설명                | Comment |
|------------|------------|---------------------|---------|
| user_id    | INTEGER    | 유저 id             |  |
| recruit_id | INTEGER    | 북마크한 채용 id    |  |
| cdate      | DATETIME   | 북마크 시점(UTC)    |  |


In [51]:
JobBookmark  = pl.read_parquet(DATA_DIR / 'JobBookmark.parquet')

JobBookmark = JobBookmark.with_columns(
    pl.col(pl.Utf8).replace(["", "nan", "NaN"], None)
)

JobBookmark

cdate,job_uuid,user_uuid
datetime[μs],str,str
2018-02-11 22:17:43,"""e9d423cb-2b66-4c9a-83a2-4f808b…","""2a4f7d22-716c-4417-b00f-e081cd…"
2018-02-20 03:38:28,"""5b54f6d1-450d-4fca-b975-726876…","""2a4f7d22-716c-4417-b00f-e081cd…"
2018-05-11 03:16:18,"""719eb06b-e616-4831-999c-e98f52…","""2a4f7d22-716c-4417-b00f-e081cd…"
2017-06-12 22:51:02,"""69420e80-c610-4d6e-b961-f1a68d…","""2a4f7d22-716c-4417-b00f-e081cd…"
2017-06-12 22:45:07,"""5af4284b-1712-49e2-81e1-a425e0…","""2a4f7d22-716c-4417-b00f-e081cd…"
…,…,…
2020-02-26 15:18:14,"""a0ec9ae9-449c-47d1-8f04-7ead33…","""29c91aed-1892-409a-804b-2b5614…"
2023-06-29 18:29:54,"""2a1f0d8f-1faa-4b1b-a0ab-cf50f0…","""b9fabe1e-8c83-46fd-bba6-7ae702…"
2024-04-03 09:54:13,"""69d41548-9a36-4442-84f1-187fc3…","""83675f11-0a6c-41ec-b634-063543…"


In [52]:
JobBookmark['cdate'].min() ,JobBookmark['cdate'].max()

(datetime.datetime(2015, 9, 25, 6, 15, 22),
 datetime.datetime(2024, 5, 6, 14, 31, 42))

북마크 시점의 최대 최소기간 / 위에서 살펴본 데이터 기간과 정확하게 일치하지 않는다. 특정시점 이후에 북마크 기능이 들어간듯

In [53]:
JobBookmark['user_uuid'].value_counts(sort=True)

user_uuid,count
str,u32
"""764148eb-ae46-4cc3-afe4-24d105…",930
"""14c2c48a-5ed6-46d4-b196-e6bfc7…",865
"""1e0fb5e0-75e9-4446-be87-dd3fb6…",781
"""d412f031-93f5-4833-8ab9-f3a5b8…",684
"""fcd13431-8f24-4825-bcad-484f8e…",559
…,…
"""29c91aed-1892-409a-804b-2b5614…",1
"""b9fabe1e-8c83-46fd-bba6-7ae702…",1
"""83675f11-0a6c-41ec-b634-063543…",1


In [54]:
JobBookmark['user_uuid'].value_counts(sort=True).describe()

statistic,user_uuid,count
str,str,f64
"""count""","""53053""",53053.0
"""null_count""","""0""",0.0
"""mean""",null,7.800991
"""std""",null,18.875778
"""min""","""00003de3-3eee-4ecd-9f34-c15c66…",1.0
"""25%""",null,1.0
"""50%""",null,3.0
"""75%""",null,7.0
"""max""","""ffffa0a5-7161-409c-873e-e39d90…",930.0


user_uuid 관점     
유저가 북마크를 했다는 시점은 취업 의사가 있는 시점으로 추측할 수 있다.    
- 북마크 숫자가 비정상적으로 많은 경우는 크롤러 / 운영진 / 헤비취업자.. 등의 이상치로 보인다.    
- 유저별 북마크 숫자는 평균 7건, 중위수는 3건이다


job_uuid 관점
- 북마크가 많이 된 공고는 많은 유저들이 관심을 가지는 공고로 볼 수 있다.
    - 그러한 공고들이 가지는 특성을 분석해보기
    - 인기 공고들을 선택한 유저의 특성 분석해보기

### 데이터 클리닝 (크롤러/봇 제거)

In [55]:
# 크롤러/봇 데이터 제거
# 1. 유저별 북마크 수 계산 (이미 하신 작업과 동일)
user_bookmark_counts = (
    JobBookmark
    .group_by("user_uuid")
    .agg(pl.len().alias("bookmark_count"))
)

# 2. 이상치 기준 설정 (예: 상위 1%를 이상치로 간주)
# quantile(0.99) 사용
threshold = user_bookmark_counts.select(pl.col("bookmark_count").quantile(0.99)).item()
print(f"이상치 판단 기준(상위 1%): {threshold}건 이상")

# 3. 크롤러/헤비유저와 일반 유저 분리
outlier_users = user_bookmark_counts.filter(pl.col("bookmark_count") >= threshold)
normal_users = user_bookmark_counts.filter(pl.col("bookmark_count") < threshold)

# 4. 분석의 정확도를 위해 원본 데이터에서 이상치 유저 제외 (클렌징된 데이터)
JobBookmark_cleaned = (
    JobBookmark
    .join(normal_users, on="user_uuid", how="inner")
)

이상치 판단 기준(상위 1%): 76.0건 이상


### Job과 join하여 직군의 인기도 분석

In [56]:
# 1. 공고별 북마크 수 집계 및 내림차순 정렬
job_popularity = (
    JobBookmark_cleaned
    .group_by("job_uuid")
    .agg(pl.len().alias("bookmarked_users_count"))
    .sort("bookmarked_users_count", descending=True)
)

# 2. 상위 인기 공고 도출 (예: 상위 100개)
top_100_jobs = job_popularity.head(100)
print(top_100_jobs)

shape: (100, 2)
┌─────────────────────────────────┬────────────────────────┐
│ job_uuid                        ┆ bookmarked_users_count │
│ ---                             ┆ ---                    │
│ str                             ┆ u32                    │
╞═════════════════════════════════╪════════════════════════╡
│ daf6711c-6458-453b-9a54-dbdca8… ┆ 744                    │
│ f7afddc2-90a5-45aa-9173-913d3b… ┆ 394                    │
│ 8e86ad58-8e37-4873-8a4d-6a009a… ┆ 315                    │
│ 2e5b3675-5752-4a69-9c3b-3190a4… ┆ 310                    │
│ d4462e55-8358-4464-a85f-133138… ┆ 293                    │
│ …                               ┆ …                      │
│ 8dbbcb48-21ba-49ca-b5f2-47634f… ┆ 93                     │
│ 9b7cf200-a47d-4f64-8e01-cefe51… ┆ 93                     │
│ 317c5eb2-dd8a-49c7-a310-adba24… ┆ 92                     │
│ 0f6a1c5f-64f1-4be2-ae2f-7bd148… ┆ 92                     │
│ 275f5363-9561-4f04-9042-b389b5… ┆ 92                     │
└───────

In [57]:
# Job 메타데이터와 join
top_jobs_characteristics = (
    top_100_jobs
    .join(Job, on="job_uuid", how="left")
)

# 어떤 직군(job_category)이 인기 있는지 그룹화하여 확인해보기
top_jobs_characteristics.group_by("job_field").agg(pl.len().alias("count")).sort("count", descending=True)


job_field,count
str,u32
"""SW 개발""",54
"""기획/PM""",18
"""디자인""",10
"""마케팅""",9
"""운영""",4
"""비즈니스""",3
"""경영지원""",2


# Application

- 지원서 마스터 테이블
- 지원서id, 유저id, 채용공고id, 지원일시	현재 하나의 채용공고에 유저가 여러번 지원할 수 있음. 채용담당자가 재지원을 요구하기도 함.

| 컬럼명   | 데이터 타입 | 설명                 | Comment |
|---------|------------|----------------------|---------|
| id      | INTEGER    | 지원 id              |  |
| user_id | INTEGER    | 유저 id              |  |
| job_id  | INTEGER    | 채용 id              |  |
| cdate   | DATETIME   | 지원서 제출시간(UTC) |  |


In [58]:
Application = pl.read_parquet(DATA_DIR / 'application.parquet')
display(Application.head())

cdate,company_uuid,job_uuid,user_uuid,application_uuid
datetime[μs],str,str,str,str
2017-02-25 23:45:01,"""de4b3596-b4ab-47cd-b8ea-6f9e14…","""459e461d-a571-4ed3-8751-8f4cb5…","""b0329bd7-fc45-4e83-993a-b73bd9…","""59c42363-f764-4cbd-aafd-20eb66…"
2020-11-23 14:46:07,"""f6156b12-d4d7-469f-84be-31799c…","""e5ed4f4a-08aa-4f2d-9042-1e9bbd…","""b0329bd7-fc45-4e83-993a-b73bd9…","""425db0e8-87f8-4b4d-8248-7f24c3…"
2017-10-28 22:30:11,"""2ed05b18-c735-474a-9faa-02095b…","""0f636d7a-53f7-485e-8a08-33253d…","""b0329bd7-fc45-4e83-993a-b73bd9…","""be4d9e85-dc53-4e70-90af-64d2d6…"
2017-07-03 17:40:05,"""3d6730ee-b869-47b7-90c6-1e074e…","""3cade56a-6354-411a-82a6-f2c777…","""b0329bd7-fc45-4e83-993a-b73bd9…","""d43879d1-c424-4192-9eae-4537fa…"
2017-02-10 10:53:39,"""de4b3596-b4ab-47cd-b8ea-6f9e14…","""459e461d-a571-4ed3-8751-8f4cb5…","""a7593a15-b93e-4bc3-91bf-e4074c…","""61873fd7-f075-44c0-90fe-f65212…"


- 공고 마감일과 제출일자 사이의 관계 (마감일에 쫓겨 제출하는가?)
- 실제 제출과 북마크는 관계가 있는지
- 채용담당자가 재지원을 요구하는 경우에 대한 케이스 확인
- 같은 회사에 여러번 다른 공고에 지원하는 경우는 얼마나 있는지 / 그러한 회사의 특징은 무엇인지

In [59]:
Application['cdate'].min() , Application['cdate'].max()

(datetime.datetime(2015, 2, 24, 10, 22, 28),
 datetime.datetime(2023, 12, 31, 14, 45, 5))

15년부터 23년까지의 데이터가 존재

In [60]:
Application.select(pl.all().null_count())


cdate,company_uuid,job_uuid,user_uuid,application_uuid
u32,u32,u32,u32,u32
0,0,0,0,0


결측치는 없다

In [61]:
Application.select(pl.all().n_unique())

cdate,company_uuid,job_uuid,user_uuid,application_uuid
u32,u32,u32,u32,u32
338701,11307,72408,36732,340730


application_uuid는 모두 유니크하다 == 지원서는 고유하다.

# log
- log	사용자 로그 테이블
- 유저id, 이벤트 발생 화면경로, 직전 화면경로, 로그 생성 일시, HTTP 요청 메소드, HTTP 응답 코드	"구직자의 로그 데이터만 제공합니다. 로그를 통해 확인 가능한 이벤트는 아래와 같습니다.

```
구직자의 로그 데이터만 제공합니다. 로그를 통해 확인 가능한 이벤트는 아래와 같습니다.

1. 플랫폼 서비스 웹 내에서의 페이지 이동 및 이벤트 전체
*단, 개인정보 이슈상 아래 이벤트를 제외
- 메시지 수신
- 메시지 송신
- 연결신청
- 연결신청 수락 등

2. 채용서비스 관련 각종 이벤트 예시
*모두 1에 포함되는 데이터입니다.
- 채용정보 조회
- 채용 기업 페이지 조회
- 채용 기업의 구성원 프로필 조회
- 지원서 업데이트 (개인정보 이슈상, 어떤 내용이 업데이트되었는지는 미제공)
- 채용공고 북마크
```



| 컬럼명         | 데이터 타입 | 설명               | Comment |
|--------------|------------|------------------|---------|
| user_id      | INTEGER    | 유저 id          |  |
| timestamp    | TIMESTAMP  | 로그 생성 시점   |  |
| date         | DATETIME   | 로그 생성일      |  |
| path         | STRING     | 이벤트 발생 화면 경로 |  |
| response_code | INTEGER    | HTTP 응답 코드   |  |
| method       | STRING     | HTTP 요청 메소드 |  |


In [96]:

Log = pl.read_parquet(DATA_DIR / 'log.parquet')

# 널값 polars용 전처리
Log = Log.with_columns(
    pl.col(pl.Utf8).replace(["", "nan", "NaN"], None)
)



In [76]:
Log.shape

(17240278, 6)

22년 데이터는 천만행

In [77]:
Log.select(pl.all().null_count())


user_uuid,URL,timestamp,date,response_code,method
u32,u32,u32,u32,u32,u32
0,637004,0,0,0,682


약 3.6% 결측치 존재

In [ ]:
# timestamp를 Datetime 타입으로 변환
# %Y-%m-%d %H:%M:%S 등 데이터 형식에 맞게 지정
# 마이크로초(%6f)와 UTC를 포함한 포맷 명시
# strict=False를 주어 혹시 모를 파싱 에러 방지
Log = Log.with_columns([
    # 1. 정밀한 timestamp 변환 (마이크로초 포함)
    pl.col("timestamp").str.to_datetime(format="%Y-%m-%d %H:%M:%S.%6f UTC", strict=False),

    # 2. date 컬럼을 Date 타입으로 변환 (시간 정보가 없으므로 Date가 효율적)
    pl.col("date").str.to_datetime(format="%Y-%m-%d", strict=False).dt.date(),

    # 3. response_code를 숫자로 변환 (200, 404 등은 Int32 혹은 Int16이면 충분) strict=False:숫자로 변환할 수 없는 경우 NULL로 지정.
pl.col("response_code").cast(pl.Int32, strict=False)])

심각한사실을 알아냄: 일부 데이터는 URL에 timestamp 정보가 삽입되어서, 열이 하나씩 밀려있는것을 확인. 
해당 문제부터 전처리하고, 가공하는게 좋겠음.

In [97]:
import polars as pl

# 1. 분리 단계 (temp 생성)
Log = Log.with_columns(
    pl.col("URL").str.splitn(",", 2).alias("temp_split")
).with_columns([
    pl.col("temp_split").struct.field("field_0").alias("temp_url"),
    pl.col("temp_split").struct.field("field_1").alias("temp_timestamp")
])

is_shifted = pl.col("response_code").cast(pl.Utf8).str.contains(r"[a-zA-Z]")

# 2. 교정 단계
Log = Log.with_columns([
    pl.when(is_shifted).then(pl.col("temp_url")).otherwise(pl.col("URL")).alias("fixed_URL"),
    pl.when(is_shifted).then(pl.col("temp_timestamp")).otherwise(pl.col("timestamp")).alias("fixed_timestamp"),
    pl.when(is_shifted).then(pl.col("timestamp")).otherwise(pl.col("date")).alias("fixed_date"),
    pl.when(is_shifted).then(pl.col("date")).otherwise(pl.col("response_code")).alias("fixed_response_code"),
    pl.when(is_shifted).then(pl.col("response_code")).otherwise(pl.col("method")).alias("fixed_method")
])

# 3. 타입 변환 단계 (🚨 수정됨)
Log = Log.with_columns([
    # " UTC" 문자열을 제거한 후 포맷 없이 변환하여 소수점 5, 6자리 모두 정상 처리되도록 변경
    pl.col("fixed_timestamp")
      .str.replace(" UTC", "")
      .str.strip_chars()
      .str.to_datetime(strict=False).alias("timestamp"),
      
    pl.col("fixed_date").str.to_datetime(format="%Y-%m-%d", strict=False).dt.date().alias("date"),
    pl.col("fixed_response_code").cast(pl.Int32, strict=False).alias("response_code"),
    pl.col("fixed_URL").alias("URL"),
    pl.col("fixed_method").alias("method")
])

# 4. 불필요한 임시/중복 컬럼 제거
Log = Log.select(["user_uuid", "URL", "timestamp", "date", "response_code", "method"])

print("전처리 완료!")

전처리 완료!


# 로그 데이터 전처리 결과물

1. timestamp, date를 각각 format에 맞게 자료형 수정
2. response_code 자료형 정수형으로 수정

In [98]:
Log.write_parquet(
    "cleaned_log_data.parquet", 
    compression="snappy"
)

print("Parquet 파일 내보내기 완료: cleaned_log_data.parquet")

Parquet 파일 내보내기 완료: cleaned_log_data.parquet


In [100]:
Log.select(pl.all().null_count())

user_uuid,URL,timestamp,date,response_code,method
u32,u32,u32,u32,u32,u32
0,637004,0,0,0,0


In [101]:
Log.shape

(17240278, 6)

In [102]:
Log.head()

user_uuid,URL,timestamp,date,response_code,method
str,str,datetime[μs],date,i32,str
"""5ce8f5ca-3476-4623-a60c-00c98e…","""@user_id""",2023-12-29 13:19:50.230356,2023-12-29,200,"""GET"""
"""5ce8f5ca-3476-4623-a60c-00c98e…","""api/users/notifications/mark_r…",2023-12-29 13:20:17.848762,2023-12-29,200,"""GET"""
"""5ce8f5ca-3476-4623-a60c-00c98e…","""jobs/id/id_title""",2023-12-29 13:22:22.277796,2023-12-29,200,"""GET"""
"""5ce8f5ca-3476-4623-a60c-00c98e…","""suggest?q=epdlxj""",2023-12-29 13:21:22.999930,2023-12-29,200,"""GET"""
"""5ce8f5ca-3476-4623-a60c-00c98e…","""api/current_guided_action/id""",2023-12-29 13:20:19.834724,2023-12-29,200,"""POST"""


In [103]:
result = (
    Log.lazy()
    .filter(pl.col("URL").is_null())
    .group_by("response_code")
    .agg(pl.count().alias("count"))
    .sort("count", descending=True)
    .collect()
)

print(result)

shape: (6, 2)
┌───────────────┬────────┐
│ response_code ┆ count  │
│ ---           ┆ ---    │
│ i32           ┆ u32    │
╞═══════════════╪════════╡
│ 200           ┆ 636289 │
│ 302           ┆ 380    │
│ 500           ┆ 233    │
│ 503           ┆ 92     │
│ 404           ┆ 8      │
│ 400           ┆ 2      │
└───────────────┴────────┘


C:\Users\user\AppData\Local\Temp\ipykernel_21360\556890044.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("count"))


결측치 데이터 만의 특성이 따로 있지는 않는것으로 보임

In [108]:
# 하루 골라서 로그 보기
target_date = "2022-11-13"
filtered_log = (
    Log.filter(pl.col("timestamp").dt.date() == pl.date(2022, 11, 13))
    .sort("timestamp")
)

print(filtered_log)

shape: (10_335, 6)
┌────────────────────┬───────────────────┬───────────────────┬────────────┬───────────────┬────────┐
│ user_uuid          ┆ URL               ┆ timestamp         ┆ date       ┆ response_code ┆ method │
│ ---                ┆ ---               ┆ ---               ┆ ---        ┆ ---           ┆ ---    │
│ str                ┆ str               ┆ datetime[μs]      ┆ date       ┆ i32           ┆ str    │
╞════════════════════╪═══════════════════╪═══════════════════╪════════════╪═══════════════╪════════╡
│ c2b030f6-d77b-4759 ┆ null              ┆ 2022-11-13        ┆ 2022-11-13 ┆ 200           ┆ GET    │
│ -8b27-532812…      ┆                   ┆ 00:00:09.498185   ┆            ┆               ┆        │
│ b84c63da-0943-40df ┆ continue?next=/co ┆ 2022-11-13        ┆ 2022-11-13 ┆ 302           ┆ GET    │
│ -8189-a6fec5…      ┆ mpanies/gamec…    ┆ 00:10:04.569430   ┆            ┆               ┆        │
│ b84c63da-0943-40df ┆ companies/company ┆ 2022-11-13        ┆ 2022-11-1

In [109]:
Log.head()

user_uuid,URL,timestamp,date,response_code,method
str,str,datetime[μs],date,i32,str
"""5ce8f5ca-3476-4623-a60c-00c98e…","""@user_id""",2023-12-29 13:19:50.230356,2023-12-29,200,"""GET"""
"""5ce8f5ca-3476-4623-a60c-00c98e…","""api/users/notifications/mark_r…",2023-12-29 13:20:17.848762,2023-12-29,200,"""GET"""
"""5ce8f5ca-3476-4623-a60c-00c98e…","""jobs/id/id_title""",2023-12-29 13:22:22.277796,2023-12-29,200,"""GET"""
"""5ce8f5ca-3476-4623-a60c-00c98e…","""suggest?q=epdlxj""",2023-12-29 13:21:22.999930,2023-12-29,200,"""GET"""
"""5ce8f5ca-3476-4623-a60c-00c98e…","""api/current_guided_action/id""",2023-12-29 13:20:19.834724,2023-12-29,200,"""POST"""


In [ ]:
import polars as pl

crosstab_result = Log.pivot(
    index="response_code",      # 행(Row) 기준
    on="method",                # 열(Column) 기준 (columns -> on 으로 변경)
    values="method",            # 집계할 대상
    aggregate_function="len"    # 데이터 개수를 세는 함수 지정
).fill_null(0)

print(crosstab_result)

C:\Users\user\AppData\Local\Temp\ipykernel_21360\2058868057.py:1: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  crosstab_result = Log.pivot(


shape: (11, 7)
┌───────────────┬──────────┬─────────┬────────┬────────┬─────────┬──────┐
│ response_code ┆ GET      ┆ POST    ┆ DELETE ┆ PUT    ┆ OPTIONS ┆ HEAD │
│ ---           ┆ ---      ┆ ---     ┆ ---    ┆ ---    ┆ ---     ┆ ---  │
│ i32           ┆ u32      ┆ u32     ┆ u32    ┆ u32    ┆ u32     ┆ u32  │
╞═══════════════╪══════════╪═════════╪════════╪════════╪═════════╪══════╡
│ 200           ┆ 12206905 ┆ 4364823 ┆ 125537 ┆ 204814 ┆ 2       ┆ 44   │
│ 302           ┆ 203046   ┆ 29      ┆ 0      ┆ 0      ┆ 0       ┆ 0    │
│ 404           ┆ 48910    ┆ 6261    ┆ 704    ┆ 10     ┆ 0       ┆ 2    │
│ 400           ┆ 1289     ┆ 57536   ┆ 83     ┆ 3836   ┆ 0       ┆ 0    │
│ 301           ┆ 59       ┆ 0       ┆ 0      ┆ 0      ┆ 0       ┆ 0    │
│ …             ┆ …        ┆ …       ┆ …      ┆ …      ┆ …       ┆ …    │
│ 500           ┆ 2820     ┆ 367     ┆ 2      ┆ 119    ┆ 0       ┆ 0    │
│ 405           ┆ 27       ┆ 1181    ┆ 1      ┆ 2      ┆ 0       ┆ 0    │
│ 401           ┆ 23   

일부 에러가 발생,
- 에러가 유독 많이 발생하는 기간이 있는지? (배포이슈?)
- 특정 api에서 에러가 많이 발생하는지

| 메서드  | 설명                                                                           |
|:------:|:-------------------------------------------------------------------------------|
| **GET**    | 서버에서 **데이터를 조회**(READ)하기 위한 요청. 엔티티 본문을 포함하지 않음.           |
| **HEAD**   | **GET**과 유사하지만, **응답의 헤더 정보만** 받음(본문은 없음). 리소스 존재 여부 확인용. |
| **POST**   | 서버에 **새로운 데이터를 생성**(CREATE)하거나, 데이터를 전송(예: 폼 제출)할 때 사용.    |
| **PUT**    | 서버에 **리소스를 생성/완전히 대체(업데이트)**. 같은 리소스에 여러 번 PUT하면 같은 결과. |
| **DELETE** | 서버의 특정 **리소스를 삭제**.                                                         |

<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>


| 코드   | 명칭                        | 설명                                                                       |
|:-----:|:---------------------------|:---------------------------------------------------------------------------|
| **200**  | OK                         | 요청이 성공적으로 처리됨. 일반적으로 **GET**, **POST** 등에 대한 성공 응답.    |
| **301**  | Moved Permanently          | **영구 리다이렉트**. 요청한 리소스가 다른 URL로 영구 이동.                     |
| **302**   | Found (Temporary Redirect) | **임시 리다이렉트**. 요청한 리소스가 임시적으로 다른 URL로 이동.               |
| **400**  | Bad Request                | 클라이언트 측의 잘못된 요청(문법 에러 등). 서버가 요청 해석 불가.               |
| **401**   | Unauthorized               | **인증**(Authentication)이 필요하지만, 유효한 인증 정보가 없음.                |
| **403**   | Forbidden                  | 클라이언트가 **접근 권한**이 없어 요청이 거부됨.                               |
| **404**   | Not Found                  | 요청한 리소스를 찾을 수 없음.                                                 |
| **405**   | Method Not Allowed         | 허용되지 않은 **HTTP 메서드**로 요청.                                         |
| **409**   | Conflict                   | 요청이 서버 상태와 **충돌**이 발생함(중복 데이터, 버전 충돌 등).               |
| **500**   | Internal Server Error      | 서버 내부 에러로 요청 처리 불가. 서버 측에서 문제 발생.                        |
| **503**   | Service Unavailable        | 서버가 현재 **서비스를 처리할 수 없음**(과부하, 점검 등 일시적 장애).         |


<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>
URL기본구조
```<scheme>://<host>/<path1>/<path2>/<path3>/<path-n>?<query>#<fragment>```

In [ ]:
## URL 구조 살펴보기
## 100만개 샘플
## 가이드 코드1
from urllib.parse import urlparse,parse_qs
sample_data = Log['URL'].dropna().sample(1000000,random_state=1).copy().reset_index(drop=True).reset_index(drop=True).to_frame()

parsing_data = pd.DataFrame(list(sample_data['URL'].map(lambda x : urlparse(x)._asdict())))
query_split = sample_data['URL'].map(lambda x : parse_qs(urlparse(x).query))
query_split.name = 'query_dict'

url_total = pd.concat([sample_data,parsing_data,query_split],axis=1)
url_total = url_total.drop(columns =['scheme','netloc','params'])
url_total.head()

```URL기본구조 <scheme>://<host>/<path1>/<path2>/<path3>/<path-n>?<query>#<fragment>```

In [ ]:
pd.set_option('display.max_colwidth',400) # 컬럼폭 늘리는 옵션, 400을 None으로 바꾸면 default 값으로 변경

In [ ]:
## URL 구조 살펴보기
## 전체데이터
from urllib.parse import urlparse,parse_qs
sample_data = Log['URL'].fillna('').reset_index(drop=True).to_frame()

parsing_data = pd.DataFrame(list(sample_data['URL'].map(lambda x : urlparse(x)._asdict())))
query_split = sample_data['URL'].map(lambda x : parse_qs(urlparse(x).query))
query_split.name = 'query_dict'

url_total = pd.concat([sample_data,parsing_data,query_split],axis=1)
url_total = url_total.drop(columns =['scheme','netloc','params'])
url_total.head()

In [ ]:
## url path를 4단계까지 구분하여 누적합 컬럼을 생성
url_total['path_1'] = url_total['path'].map(lambda x : x.split('/')[0] if '/' in x else None)
url_total['path_2'] = url_total['path'].map(lambda x : '/'.join(x.split('/')[:2]) if len(x.split('/')) >=2  else None)
url_total['path_3'] = url_total['path'].map(lambda x : '/'.join(x.split('/')[:3]) if len(x.split('/')) >=3  else None)
url_total['path_4'] = url_total['path'].map(lambda x : '/'.join(x.split('/')[:4]) if len(x.split('/')) >=4  else None)

In [ ]:
Log.head()

In [ ]:
log22_total = pd.concat([Log,url_total.drop(columns =['URL'])],axis=1) # 메모리 엄청 먹음, 일부만 샘플해서 살펴보기를 추천

불필요 변수제거,
메모리 제거

In [ ]:
%xdel url_total

In [ ]:
%xdel log22

In [ ]:
log22_total.shape

In [ ]:
# 30분 기준으로 세션을 생성하여 유저의 행동 분석을 진행하려함
# section_preprocessing_df 데이터 프레임을 생성하여 세션만 부여, 이후 전체데이터에 옮긴 후 해당데이터 삭제
THRESHOLD = pd.Timedelta(minutes=30)
section_preprocessing_df = log22_total[['user_uuid','datetime']].copy()
section_preprocessing_df['time_diff'] = section_preprocessing_df.groupby('user_uuid')['datetime'].diff()
section_preprocessing_df['new_session'] = (section_preprocessing_df['time_diff'] >= THRESHOLD) | (section_preprocessing_df['time_diff'].isna())
section_preprocessing_df['session_id'] = section_preprocessing_df.groupby('user_uuid')['new_session'].cumsum()

In [ ]:
section_preprocessing_df.head(4)

In [ ]:
log22_total['session_id'] = section_preprocessing_df['session_id']

In [ ]:
%xdel section_preprocessing_df

### 세션분석

In [ ]:
sec = log22_total[['user_uuid','datetime','URL','path','session_id']].copy()
sec['session_unique'] = sec['user_uuid'] + '_' + sec['session_id'].astype('str')
sec.head()

In [ ]:
sec['session_unique'].value_counts().describe()

세션의 경우 중위수가 6개의 로그밖에 없다. -> 대체로 무시해도 되는 로그가 많다는 것

In [ ]:
# 세션 샘플 볼 수 있는 코드
# 일부 특징적인 케이스 아래에 별도 표기
smp = sec.sample(1).session_unique.values[0]
print(smp)
session_sample = sec[sec.session_unique == smp]
print(session_sample.shape)
session_sample[['datetime','URL','path','session_id']].reset_index(drop=True)

In [ ]:
def display_session(user_session_unique):
    '''
    세션id 넣으면 세션 로그 출력하는 함수
    '''
    session_sample = sec[sec.session_unique == user_session_unique]
    print(session_sample.shape)
    display(session_sample[['datetime','URL','path','session_id']].reset_index(drop=True))
    return

In [ ]:
# 로그는 검색단어 하나씩 들어갈 때 마다 입력이 됨. suggest가 포함된 URL은 마지막 정보만 있어도 될듯
display_session('83b4586c-1237-4f07-9dff-95353eb1ffe6_4')

In [ ]:
# 실제 지원하는 유저
display_session('4f9fd8a8-1dec-4364-8d83-f780ff5117f4_14')

In [ ]:
# 여러 잡타이틀을 보고 북마크
display_session('c6f0289b-7974-4a08-a912-77dc0636e991_14')

In [ ]:
# 개발자 직무 희망자의 로그
display_session('a3ce9752-16f4-42c6-b797-25770e1aec7b_1')

로그 분석방향 제안
- 로그 자체를 줄일 수 있다
    - 검색의 경우 마지막 입력값에 해당하는 로그만 (suggest, search 등)
    - template 관련한 부분도 불필요해 보임
- 정의된 세션을 통한 유저 행동에 대한 분석 (방문주기, 세션 유지 시간, 행동패턴 등)


api 구조를 보기 위해
- 웹 URL , api 라우터, query string 살펴보기
- 구조를 다양한 방식으로 파싱해서 유저의 행동 추적해보기